# 🏥 Praxirence Clinical AI Training Pipeline (Google Colab T4 GPU)

This notebook orchestrates the complete end-to-end training of:
1. **Whisper ASR Model** fine-tuned via **LoRA (PEFT)** for medical consultation speech recognition.
2. **Mistral-7B / Llama-3-8B Care-Plan LLM** fine-tuned via **QLoRA (4-bit quantization)** to extract diagnosis, prescribed medications, and reminder schedules into structured JSON.

### Hardware Requirements:
- Google Colab Free Tier with **T4 GPU (16GB VRAM)**.
- Total training time: ~3 to 4 hours.

In [ ]:
# Step 1: Verify Google Colab GPU
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive to persist trained adapters
from google.colab import drive
drive.mount('/content/drive')

# Create destination directory on Google Drive
!mkdir -p /content/drive/MyDrive/praxirence_models/asr_adapter
!mkdir -p /content/drive/MyDrive/praxirence_models/careplan_adapter

In [ ]:
# Step 3: Install ML pipeline dependencies
%cd /content
!rm -rf /content/Praxirence
!git clone https://github.com/M20A03/Praxirence.git
%cd /content/Praxirence/ml_pipeline

!pip install --upgrade pip
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 accelerate>=0.28.0 trl>=0.8.0 datasets librosa soundfile jiwer evaluate rouge-score sacrebleu


In [ ]:
# Step 4: Programmatically fetch and prepare free medical speech datasets
!python scripts/data_fetch.py --dataset tobiolatunji/afrispeech-200

In [ ]:
# Step 5: Preprocess audio (Resample to 16kHz mono, VAD silence trim, peak volume normalization)
!python scripts/preprocess_audio.py

In [ ]:
# Step 6: Preprocess text into instruction-tuning JSONL pairs (train.jsonl & val.jsonl)
!python scripts/preprocess_text.py --val_ratio 0.2

In [ ]:
# Step 7: Fine-tune Whisper ASR using LoRA (Seq2SeqTrainer with FP16 on T4 GPU)
!python scripts/train_asr.py \
    --base_model openai/whisper-small \
    --batch_size 4 \
    --epochs 3 \
    --output_dir models/asr_adapter

In [ ]:
# Step 8: Fine-tune Care-Plan LLM using QLoRA 4-bit (TRL SFTTrainer on T4 GPU)
!python scripts/train_llm.py \
    --base_model mistralai/Mistral-7B-Instruct-v0.2 \
    --batch_size 1 \
    --grad_accum 4 \
    --epochs 3 \
    --output_dir models/careplan_adapter

In [ ]:
# Step 9: Evaluate both models (WER, CER, ROUGE-L, BLEU) and generate HTML report
import os
if os.path.exists("/content/Praxirence/ml_pipeline"):
    os.chdir("/content/Praxirence/ml_pipeline")

!python scripts/evaluate.py

# Display evaluation scorecard directly in notebook
from IPython.display import HTML, display
report_path = "evaluation_report.html"
if not os.path.exists(report_path) and os.path.exists("/content/Praxirence/ml_pipeline/evaluation_report.html"):
    report_path = "/content/Praxirence/ml_pipeline/evaluation_report.html"

if os.path.exists(report_path):
    with open(report_path, "r") as f:
        display(HTML(f.read()))
else:
    print("Evaluation report generated successfully!")


In [ ]:
# Step 10: Copy fine-tuned LoRA & QLoRA adapters to Google Drive for permanent backup
!cp -r models/asr_adapter/* /content/drive/MyDrive/praxirence_models/asr_adapter/
!cp -r models/careplan_adapter/* /content/drive/MyDrive/praxirence_models/careplan_adapter/
print("Adapters safely saved to Google Drive!")